# Допуск: DataSentinel: A Game-Theoretic Detection of Prompt Injection Attacks

**Статья:** Y. Liu и др. — [arXiv:2504.11358v4](https://arxiv.org/abs/2504.11358v4),
12.11.2025. Номера разделов, формул и таблиц относятся к [PDF v4](https://arxiv.org/pdf/2504.11358v4).

5 заданий. Под заголовком каждого указан его тип:

- **вопрос по коду** — код корректен, ответьте на вопросы через код и статью;
- **расхождение со статьёй** — найдите одно отклонение от метода;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду.

Для отклонения укажите место, объясните, что не так, приведите якорь в статье
и опишите исправление словами. Писать, запускать и отлаживать код не нужно.
Вопросы относятся только к заданиям без отклонений.

Общие импорты:

In [ ]:
from copy import deepcopy
import torch

`detector` — отдельная detection LLM g; `backend` — основная LLM f, выполняющая
целевую задачу приложения. Непоказанные функции считайте корректными. Сниппеты —
сокращённые модели механизма, не полное обучение Mistral или реализация GCG.

## Задание 1. Ответ известен заранее

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

`make_instruction(key)` возвращает инструкцию детектора, требующую повторить
`key`, игнорируя следующий текст. `detector(prompt)` возвращает только сгенерированное
продолжение, без входного промпта. Результат `True` означает «contaminated»
(§4.2, формула 1, рис. 1 и 6).

In [ ]:
def detect(data, key, detector, make_instruction):
    instruction = make_instruction(key)
    outputs = [detector(instruction + text) for text in data]
    contaminated = torch.tensor([key not in output for output in outputs])  # (B,)
    return contaminated

**Вопросы.**

1. Ключ — `K7Q2M9Z`, три сгенерированных ответа — `K7Q2M9Z`,
   `K7Q2M9Z. Done.`, `positive`. Каковы три вердикта? Что именно проверяет
   формула 1: наличие ключа или точное равенство ответа ключу (§4.2)?
2. Из вердикта детектора следует ли, что backend уже выполнил задачу атакующего?
   Почему выход g нельзя подменить ответом f? Какая информация о ключе доступна
   реальному атакующему и что дополнительно разрешено гипотетическому противнику
   при обучении (§3.1, §4.3, §6)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Обновить детектор

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Внешний min-шаг. `key_nll(detector, batch)` возвращает `(B,)` — для каждого текста
сумму отрицательных log-probability токенов секретного ключа под g при входе
`detection_instruction || text`. `optimizer` обновляет только g. `contaminated`
и `clean` — два непустых minibatch; β положительна. Найдите одно отклонение
от полной цели обновления (§4.3, формула 3; §4.4, алгоритм 3).

In [ ]:
def update_detector(detector, contaminated, clean, optimizer, key_nll, beta):
    optimizer.zero_grad()
    loss = -key_nll(detector, contaminated).mean()
    loss.backward()
    optimizer.step()
    return float(loss.detach())

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Выбрать сильную атаку

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Упрощённый выбор среди K готовых кандидатов `x_c` для одной пары target/injected tasks.
`detector_nll` — loss секретного ключа под g на каждом кандидате, `backend_nll` —
loss желаемого атакующим ответа y_e под f на том же кандидате. Оба тензора имеют
форму `(K,)`; α положительна. Найдите одно отклонение от двух целей внутреннего
max-шага (§4.3, формула 2). Здесь требуется только ранжирование готового списка,
а не реализация оптимизации токенов.

In [ ]:
def choose_attack(candidates, detector_nll, backend_nll, alpha):
    candidate_cost = detector_nll
    index = int(candidate_cost.argmin())
    return candidates[index]

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Несколько раундов игры

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`optimize_separator(g, backend, target, injected)` ищет разделитель z против
переданной g; начальные условия оптимизации здесь скрыты внутри корректной функции.
`update(g, contaminated, target)` выполняет полный внешний min-шаг и меняет g
на месте. `split_tasks` выбирает одну задачу как injected, остальные как target;
`compose` собирает x_t || z || s_e || x_e. Backend на всём протяжении фиксирован.
Найдите одно отклонение в чередовании обучения (§4.4, рис. 2, алгоритм 1).

In [ ]:
def train(detector, backend, tasks, rounds, split_tasks,
          optimize_separator, compose, update):
    initial_detector = deepcopy(detector)
    for _ in range(rounds):
        target, injected = split_tasks(tasks)
        separator = optimize_separator(initial_detector, backend, target, injected)
        contaminated = compose(target, injected, separator)
        update(detector, contaminated, target)
    return detector

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Где перестаёт помогать обучение

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

FNR DataSentinel Minimax из табл. 6. Строки — duplicate sentence detection,
grammar correction, hate detection, natural language inference, sentiment analysis,
spam detection, summarization. Столбцы — adaptive Heuristic-I / Heuristic-II /
optimization-based. Injected task во всех строках — sentiment analysis.
Найдите один неверный комментарий (§5.4.2, табл. 6, §6).

In [ ]:
def adaptive_fnr():
    fnr = torch.tensor([[0.00, 0.00, 0.00],
                        [0.00, 0.00, 0.03],
                        [0.00, 0.00, 0.06],
                        [0.00, 0.00, 0.00],
                        [0.00, 0.00, 0.87],
                        [0.00, 0.00, 0.01],
                        [0.00, 0.00, 0.04]])              # (7, 3), fractions
    average_fnr = fnr.mean(dim=0)
    # Minimax keeps FNR low even when target and injected tasks have the same type.
    return fnr, average_fnr

**Ваш ответ:**

_Напишите ответ здесь._